1.Install Required Libraries

In [ ]:
!pip install transformers
!pip install torch
!pip install pandas
!pip install scikit-learn

2.Import Libraries

In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

3.Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving dataset.csv to dataset.csv


**4.Data Preprocessing**

In [ ]:
import re

df = pd.read_csv("dataset.csv")

# Select useful columns
df = df[['Score','Summary']]

# Convert rating to sentiment
def convert_sentiment(score):
    if score >= 4:
        return 1
    elif score <= 2:
        return 0
    else:
        return None

df['sentiment'] = df['Score'].apply(convert_sentiment)

# Remove neutral reviews
df = df.dropna()

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df['review'] = df['Summary'].apply(clean_text)

df = df[['review','sentiment']]

# Reduce dataset size for faster training
df = df.sample(20000)

df.head()


,review,sentiment
223503,great value,1.0
443243,the captain,1.0
410389,starbucks,1.0
63965,sunmaid honey raisen bran muffin mixgreat muffins,1.0
175232,mothers iced oatmeal cookies,1.0


5️⃣ Train/Test Split

In [ ]:
texts = df["review"].tolist()
labels = df["sentiment"].astype(int).tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2
)

In [ ]:
6️⃣ Load BERT Tokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

7️⃣ Create Dataset Class

In [ ]:
class SentimentDataset(Dataset):

    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=128
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

8️⃣ Create DataLoader

In [ ]:
train_dataset = SentimentDataset(train_texts, train_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

9️⃣ Load BERT Model

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-5
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🔟 Train Model

In [ ]:
model.train()

for epoch in range(2):

    total_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print("Epoch", epoch+1, "completed | Loss:", total_loss)

1️⃣1️⃣ Save Model

In [ ]:
torch.save(model.state_dict(), "bert_sentiment.pt")
print("Model saved successfully!")

In [5]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Load model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

1️⃣2️⃣ Prediction Function

In [7]:
def predict(text):

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits)

    if prediction == 1:
        return "Positive"
    else:
        return "Negative"

1️⃣3️⃣ Test the Model

In [8]:
text = "This product tastes amazing"

print("Sentiment:", predict(text))

Sentiment: Positive


Dataset
   ↓
Text Preprocessing
   ↓
BERT Tokenization
   ↓
Transformer Encoder
   ↓
Fine-tuning
   ↓
Sentiment Prediction